# 20. Verificaciones posteriores al protocolo

Este capítulo revisa las decisiones que pueden objetarse y los puntos que quedaron abiertos. El objetivo es respaldar los resultados de los capítulos 13 y 16 o, si alguna decisión los cambia, mostrarlo. Son nueve verificaciones:

| # | Verificación | Pregunta |
|---|---|---|
| V1 | Reproducibilidad | ¿Evaluar de nuevo las predicciones guardadas da exactamente las cifras del capítulo 13? |
| V2 | Orden de las filas en el SVM | ¿La calibración interna del SVM (`CalibratedClassifierCV` con `cv=3`, sin barajar) se vio afectada por el orden de las filas, como ocurrió en Bank Marketing (capítulo 16)? |
| V3 | Diferencia de reproducción de Bayes | ¿De dónde viene la diferencia de hasta 1.2e-3 entre el Bayes del capítulo 13 y el del capítulo 12, y afecta a DES-CPL? |
| V4 | Conjunto de modelos ampliado | ¿Gana DES-CPL si combina también a LightGBM y CatBoost? (exploratoria) |
| V5 | Pliegues del conjunto de selección | ¿Cambia la conclusión si las predicciones fuera de pliegue se calculan con 5 pliegues en lugar de 3? |
| V6 | Predicción conforme | ¿La subcobertura de los readmitidos (0.869 frente a 0.90 en la sección 13.8) se debe a la diferencia entre los modelos de pliegue y el modelo final? |
| V7 | Parada temprana de XGBoost | ¿Cambia XGBoost si su validación interna se agrupa por paciente y se separa antes de ADASYN? |
| V8 | Calibración del SVM | ¿Cambia el SVM si los pliegues de su calibración se agrupan por paciente? |
| V9 | Duplicados en los conjuntos adicionales | ¿Cambian las conclusiones del capítulo 16 sin los registros que tienen una copia exacta en el entrenamiento? |

**Estatus.** Ninguna de estas verificaciones forma parte del protocolo pre-registrado (`protocolo-v1`). Son análisis de sensibilidad posteriores, y no cambian el veredicto del capítulo 17 sea cual sea su resultado. Las reglas de decisión se fijaron antes de ejecutarlas y se escriben aquí para que no se ajusten a los resultados:

| # | La conclusión se mantiene si... |
|---|---|
| V1 | todas las métricas y los hiperparámetros elegidos coinciden con los del capítulo 13 (diferencia < 1e-9). |
| V2 | al barajar las filas, la AUC-PR del SVM cambia en promedio menos de 0.002 y en ningún fold más de 0.005. Una variación pequeña es normal, porque barajar cambia los pliegues de la calibración. El fallo de Bank Marketing fue de 0.30. |
| V3 | quitar a Bayes del conjunto cambia la AUC-PR media de DES-CPL en menos de 0.0005. Esa es una cota superior del efecto de su diferencia de reproducción. |
| V4 | DES-CPL con 9 modelos no supera a XGBoost por al menos 0.006 con p < 0.05 (Nadeau-Bengio). Ese umbral es lo que haría falta para que la prueba DeLong en el conjunto de prueba fuera significativa (sección 13.8). |
| V5 | con 5 pliegues la AUC-PR media de DES-CPL cambia menos de 0.002 y la prueba de Nadeau-Bengio frente a XGBoost no cambia de veredicto a α = 0.05. |
| V6 | (descriptiva) se reporta la cobertura de los readmitidos con 3 y con 5 pliegues. |
| V7 | la AUC-PR media de XGBoost cambia menos de 0.0024 (la cota declarada) y el veredicto de Nadeau-Bengio de DES-CPL frente a XGBoost no cambia. |
| V8 | la AUC-PR media del SVM cambia menos de 0.002. |
| V9 | las diferencias de DES-CPL frente a XGBoost y al mejor benchmark de cada conjunto conservan su signo y cambian menos de 0.002. |

V7 a V9 se añadieron después de ejecutar V1 a V6, con sus reglas escritas antes de ejecutarlas.

**Ejecución.** Sigue el patrón de los capítulos 12 y 13. Cada celda lee primero su archivo de `results/verificaciones/` y calcula solo lo que falta. Si se interrumpe, continúa donde quedó, y para rehacer una parte basta con borrar su archivo. Las verificaciones van de la más barata a la más costosa. En el equipo de los autores (4 núcleos), el capítulo completo debería tomar entre 1.5 y 2.5 horas: cerca de la mitad es V5. Si ya está todo calculado, corre en segundos.

In [1]:
from config import *
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedGroupKFold

from src import configuracion as cfg
from src import experimento_des_cpl as exd
from src.balanceo import construir_pipeline
from src.estadistica import prueba_nadeau_bengio
from src.incertidumbre import evaluar_conformal
from src.sota import construir_pipeline_sota

cfg.N_JOBS = 4
FOLDS = list(range(5))
VERIF = RESULTADOS / "verificaciones"
VERIF.mkdir(exist_ok=True)
K3 = RESULTADOS / "des_cpl_60"       # predicciones del capítulo 13 (3 pliegues)

## 20.1. Datos y configuración

Se usan los mismos datos, particiones y configuraciones que en la corrida definitiva del capítulo 13:
- el entrenamiento del ETL;
- las 5 particiones externas del benchmark (capítulos 8 a 11) (`StratifiedGroupKFold`, semilla 42);
- para cada modelo base y fold, la configuración de Random Search con 60 evaluaciones del capítulo 12;
- para LightGBM y CatBoost, la configuración de `results/benchmarks_sota.csv`.

In [2]:
roles = roles_variables()
predictores = roles["numericas"] + roles["ordinales"] + roles["binarias"] + roles["categoricas"]
train = leer_tabla("diabetes_train")
X, y, grupos = train[predictores], train[roles["objetivo"]], train[roles["identificador"]]
razon = float((1 - y.mean()) / y.mean())
mejores = exd.con_presupuesto(
    exd.mejores_combinaciones(pd.read_csv(RESULTADOS / "tabla_maestra_clasificacion.csv")),
    RESULTADOS / "convergencia_evaluaciones.csv", RESULTADOS / "convergencia_reajustes.csv", 60)
particiones = exd.particiones_externas(X, y, grupos)
oficial = pd.read_csv(RESULTADOS / "des_cpl_evaluacion_60.csv")
sota = pd.read_csv(RESULTADOS / "benchmarks_sota.csv")
print(f"{len(X):,} encuentros y {grupos.nunique():,} pacientes de entrenamiento; "
      f"{len(FOLDS)} folds externos; prevalencia {y.mean():.4f}")

79,473 encuentros y 56,036 pacientes de entrenamiento; 5 folds externos; prevalencia 0.1139


In [3]:
def leer_fold(carpeta, fold):
    """Conjunto de selección y fold de evaluación guardados de un fold externo."""
    carpeta = Path(carpeta)
    return (pd.read_csv(carpeta / f"seleccion_fold{fold}.csv"),
            pd.read_csv(carpeta / f"evaluacion_fold{fold}.csv"))


def evaluar_guardado(carpeta, nombre, modelos=exd.MODELOS, variantes=("DES-CPL",)):
    """Evalúa todos los métodos en cada fold una sola vez y guarda el resultado.

    Usa ``exd.evaluar_fold`` con la misma semilla por fold que el capítulo 13
    (``SEMILLA + fold``). Solo se evalúa la variante final de DES-CPL: cada
    variante tiene su propia búsqueda con la misma semilla, así que omitir
    las ablaciones no cambia su resultado (lo comprueba V1). Guarda también
    las predicciones de DES-CPL en el fold de evaluación y fuera de pliegue
    en el conjunto de selección, que V6 usa para la predicción conforme.
    """
    ruta = VERIF / f"{nombre}.csv"
    tabla = pd.read_csv(ruta) if ruta.exists() else pd.DataFrame()
    for fold in FOLDS:
        ruta_pred = VERIF / f"{nombre}_predicciones_fold{fold}.csv"
        ruta_cal = VERIF / f"{nombre}_calibracion_fold{fold}.csv"
        hecho = len(tabla) and fold in set(tabla["fold"])
        if hecho and ruta_pred.exists() and ruta_cal.exists():
            continue
        seleccion, evaluado = leer_fold(carpeta, fold)
        inicio = time.perf_counter()
        t, pred = exd.evaluar_fold(seleccion, evaluado, list(modelos),
                                   semilla=cfg.SEMILLA + fold,
                                   devolver_predicciones=True, variantes=variantes)
        t.insert(0, "fold", fold)
        pd.DataFrame({"y": evaluado["y"], "DES-CPL": pred["DES-CPL"]}).to_csv(ruta_pred, index=False)
        pd.DataFrame({"y": seleccion["y"],
                      "DES-CPL": pred["DES-CPL (fuera de pliegue)"]}).to_csv(ruta_cal, index=False)
        previa = tabla[tabla["fold"] != fold] if len(tabla) else tabla
        tabla = pd.concat([previa, t], ignore_index=True)
        tabla.to_csv(ruta, index=False)
        print(f"[listo] {nombre}, fold {fold}: {time.perf_counter() - inicio:.0f} s")
    return tabla.sort_values("fold", kind="stable").reset_index(drop=True)


def por_fold(tabla, metodo, columna="auc_pr"):
    """Serie por fold externo de una métrica de un método."""
    return tabla[tabla["método"] == metodo].set_index("fold")[columna].reindex(FOLDS)


def comparar(tabla_a, a, tabla_b=None, b="xgboost"):
    """Diferencia por fold (a − b) con la t de Nadeau-Bengio unilateral."""
    tabla_b = tabla_a if tabla_b is None else tabla_b
    d = (por_fold(tabla_a, a) - por_fold(tabla_b, b)).to_numpy()
    nb = prueba_nadeau_bengio(d)
    return {"comparación": f"{a} − {b}", "dif. media": d.mean(),
            "gana en folds": f"{int((d > 0).sum())}/{len(d)}",
            "t (Nadeau-Bengio)": nb["t"], "p (unilateral)": nb["p"]}


def comparar_arrays(a, b):
    """Como ``comparar``, pero con dos vectores por fold."""
    d = np.asarray(a) - np.asarray(b)
    nb = prueba_nadeau_bengio(d)
    return {"dif. media": d.mean(), "gana en folds": f"{int((d > 0).sum())}/{len(d)}",
            "t (Nadeau-Bengio)": nb["t"], "p (unilateral)": nb["p"]}


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

## 20.2. V1: reproducibilidad de la evaluación

Se vuelve a evaluar, desde cero, cada fold con las predicciones guardadas del capítulo 13 (`results/des_cpl_60/`) y se compara con `results/des_cpl_evaluacion_60.csv`. Esta evaluación sirve además de referencia, con 3 pliegues, para V3 a V6.

In [4]:
k3 = evaluar_guardado(K3, "evaluacion_k3")
cruce = k3.merge(oficial, on=["fold", "método"], suffixes=("", "_oficial"))
diferencias = pd.Series({c: (cruce[c] - cruce[f"{c}_oficial"]).abs().max()
                         for c in ["auc_pr", "auc_roc", "brier", "auc_pr_seleccion",
                                   "k", "tau", "lambda"]}, name="máx. |diferencia|")
mismos_modelos = (cruce["modelos_combinados"].fillna("")
                  == cruce["modelos_combinados_oficial"].fillna("")).all()
v1_cumple = bool(len(cruce) == len(k3) and mismos_modelos
                 and (diferencias.fillna(0) < 1e-9).all())
display(diferencias.to_frame())
print(f"Métodos comparados: {cruce['método'].nunique()} en {cruce['fold'].nunique()} folds; "
      f"mismos modelos combinados: {mismos_modelos}")
print("V1:", "se reproduce exactamente" if v1_cumple else "NO se reproduce: revisar antes de seguir")

,máx. |diferencia|
auc_pr,0.0
auc_roc,0.0
brier,0.0
auc_pr_seleccion,0.0
k,0.0
tau,0.0
lambda,0.0


Métodos comparados: 11 en 5 folds; mismos modelos combinados: True
V1: se reproduce exactamente


## 20.3. V2: orden de las filas y calibración del SVM

En Bank Marketing (capítulo 16) el SVM falló porque los datos venían ordenados por fecha y la calibración interna de `CalibratedClassifierCV(cv=3)` usa pliegues contiguos, sin barajar. Así, entrenaba y calibraba en épocas distintas. El problema principal también tiene un orden: las filas siguen el orden del archivo original. Para descartar el mismo efecto se mira primero si la prevalencia cambia a lo largo de las filas. Después se reajusta el SVM final de cada fold con las filas del entrenamiento barajadas y se compara con el original.

In [5]:
posicion = np.arange(len(y))
tercios = pd.DataFrame({"tercio de filas": ["1.º", "2.º", "3.º"],
                        "prevalencia": [y.iloc[i].mean() for i in np.array_split(posicion, 3)]})
display(tercios.round(4))
if "encounter_id" in train:
    rho = stats.spearmanr(posicion, train["encounter_id"]).statistic
    print(f"Correlación de Spearman entre la posición de la fila y encounter_id: {rho:.3f}")

ruta = VERIF / "svm_orden.csv"
if ruta.exists():
    svm_orden = pd.read_csv(ruta)
else:
    fila, filas = mejores.loc["svm"], []
    for fold in FOLDS:
        entrenamiento, evaluacion = particiones[fold]
        _, evaluado = leer_fold(K3, fold)
        assert np.array_equal(evaluado["fila"], evaluacion)
        base = construir_pipeline("svm", fila["balanceo"], roles, cfg.SEMILLA, razon).set_params(
            **fila["hiperparametros_por_fold"][fold])
        barajado = np.random.default_rng(cfg.SEMILLA + fold).permutation(entrenamiento)
        p = clone(base).fit(X.iloc[barajado], y.iloc[barajado]).predict_proba(X.iloc[evaluacion])[:, 1]
        filas.append({"fold": fold,
                      "AUC-PR orden original": average_precision_score(evaluado["y"], evaluado["final_svm"]),
                      "AUC-PR filas barajadas": average_precision_score(evaluado["y"], p)})
    svm_orden = pd.DataFrame(filas)
    svm_orden["diferencia"] = svm_orden["AUC-PR filas barajadas"] - svm_orden["AUC-PR orden original"]
    svm_orden.to_csv(ruta, index=False)
v2_cumple = bool(abs(svm_orden["diferencia"].mean()) < 0.002 and (svm_orden["diferencia"].abs() < 0.005).all())
display(svm_orden.round(4))
print(f"Diferencia media {svm_orden['diferencia'].mean():+.4f}; máx. |dif.| {svm_orden['diferencia'].abs().max():.4f}")
print("V2:", "el orden de las filas no afecta al SVM" if v2_cumple else "el orden SÍ afecta al SVM")

,tercio de filas,prevalencia
0,1.º,0.1169
1,2.º,0.1155
2,3.º,0.1092


,fold,AUC-PR orden original,AUC-PR filas barajadas,diferencia
0,0,0.2373,0.2374,0.0001
1,1,0.2054,0.2055,0.0001
2,2,0.2091,0.2091,0.0000
3,3,0.2058,0.2063,0.0005
4,4,0.2147,0.2161,0.0014


Diferencia media +0.0004; máx. |dif.| 0.0014
V2: el orden de las filas no afecta al SVM


## 20.4. V3: diferencia de reproducción de Bayes

En el control del capítulo 13, seis de los siete modelos base reproducen la AUC-PR del capítulo 12 con diferencias menores que 1e-6. Bayes difiere hasta en 1.2e-3 por fold. Se reajusta el mismo Bayes de dos formas, con `src/` y con `docs/experimento.py` (el motor con que se calculó el capítulo 12), para localizar la diferencia. Después se acota su efecto evaluando DES-CPL sin Bayes en el conjunto: si quitarlo del todo apenas cambia el resultado, una diferencia de 1e-3 en sus probabilidades tampoco puede cambiarlo.

In [6]:
import experimento as ex

ruta = VERIF / "bayes_reproduccion.csv"
if ruta.exists():
    bayes = pd.read_csv(ruta)
else:
    fila, filas = mejores.loc["bayes"], []
    for fold in FOLDS:
        entrenamiento, evaluacion = particiones[fold]
        parametros = fila["hiperparametros_por_fold"][fold]
        _, evaluado = leer_fold(K3, fold)
        X_tr, y_tr, X_ev = X.iloc[entrenamiento], y.iloc[entrenamiento], X.iloc[evaluacion]
        p_src = (construir_pipeline("bayes", fila["balanceo"], roles, cfg.SEMILLA, razon)
                 .set_params(**parametros).fit(X_tr, y_tr).predict_proba(X_ev)[:, 1])
        p_ex = (ex.construir_pipeline("bayes", fila["balanceo"], roles, ex.SEMILLA, razon)
                .set_params(**parametros).fit(X_tr, y_tr).predict_proba(X_ev)[:, 1])
        combinados = oficial.query("fold == @fold and `método` == 'DES-CPL'")["modelos_combinados"].iloc[0]
        filas.append({"fold": fold,
                      "AUC-PR cap. 12": fila["auc_pr_por_fold"][fold],
                      "AUC-PR guardada cap. 13": average_precision_score(evaluado["y"], evaluado["final_bayes"]),
                      "AUC-PR src/ ahora": average_precision_score(evaluado["y"], p_src),
                      "AUC-PR experimento.py ahora": average_precision_score(evaluado["y"], p_ex),
                      "máx. |p src − p guardada|": np.abs(p_src - evaluado["final_bayes"]).max(),
                      "máx. |p experimento − p src|": np.abs(p_ex - p_src).max(),
                      "Bayes en DES-CPL": "bayes" in combinados.split(", ")})
    bayes = pd.DataFrame(filas)
    bayes["dif. con cap. 12"] = bayes["AUC-PR guardada cap. 13"] - bayes["AUC-PR cap. 12"]
    bayes.to_csv(ruta, index=False)
display(bayes.round(6))

# Las predicciones guardadas se escribieron con 8 cifras significativas.
estable = bayes["máx. |p src − p guardada|"].max() < 1e-6
mismo_motor = bayes["máx. |p experimento − p src|"].max() < 1e-9
if estable and mismo_motor:
    print("src/ y experimento.py dan hoy las mismas probabilidades que las guardadas: la diferencia con el "
          "capítulo 12 no está en el código del pipeline, sino en lo que recibió aquella corrida "
          "(datos de entrada o parámetros).")
elif not mismo_motor:
    print("src/ y experimento.py dan probabilidades distintas para Bayes: la diferencia está en el pipeline.")
else:
    print("El reajuste de hoy no reproduce las probabilidades guardadas: Bayes no es determinista en este entorno.")

,fold,AUC-PR cap. 12,AUC-PR guardada cap. 13,AUC-PR src/ ahora,AUC-PR experimento.py ahora,máx. |p src − p guardada|,máx. |p experimento − p src|,Bayes en DES-CPL,dif. con cap. 12
0,0,0.218595,0.217828,0.218595,0.218595,0.0,0.0,False,-0.000767
1,1,0.191269,0.190897,0.191269,0.191269,0.0,0.0,False,-0.000372
2,2,0.191097,0.190402,0.191097,0.191097,0.0,0.0,False,-0.000695
3,3,0.192545,0.192665,0.192545,0.192545,0.0,0.0,True,0.000120
4,4,0.205973,0.206608,0.205973,0.205973,0.0,0.0,True,0.000635


src/ y experimento.py dan hoy las mismas probabilidades que las guardadas: la diferencia con el capítulo 12 no está en el código del pipeline, sino en lo que recibió aquella corrida (datos de entrada o parámetros).


In [7]:
sin_bayes = evaluar_guardado(K3, "evaluacion_sin_bayes", modelos=[m for m in exd.MODELOS if m != "bayes"])
efecto_bayes = pd.DataFrame({"DES-CPL con Bayes": por_fold(k3, "DES-CPL"),
                             "DES-CPL sin Bayes": por_fold(sin_bayes, "DES-CPL")})
efecto_bayes["diferencia"] = efecto_bayes["DES-CPL sin Bayes"] - efecto_bayes["DES-CPL con Bayes"]
efecto_bayes["Bayes en DES-CPL"] = bayes.set_index("fold")["Bayes en DES-CPL"]
v3_cumple = bool(abs(efecto_bayes["diferencia"].mean()) < 0.0005)
display(efecto_bayes.round(4))
print(f"Efecto medio de quitar a Bayes: {efecto_bayes['diferencia'].mean():+.5f}")
print("V3:", "la diferencia de Bayes no puede afectar a DES-CPL" if v3_cumple
      else "Bayes influye en DES-CPL más de lo previsto")

,DES-CPL con Bayes,DES-CPL sin Bayes,diferencia,Bayes en DES-CPL
fold,,,,
0,0.2562,0.2545,-0.0017,False
1,0.2189,0.2189,0.0000,False
2,0.2128,0.2135,0.0007,False
3,0.2192,0.2192,0.0001,True
4,0.2325,0.2282,-0.0043,True


Efecto medio de quitar a Bayes: -0.00105
V3: Bayes influye en DES-CPL más de lo previsto


## 20.5. V4: DES-CPL con un conjunto de modelos ampliado (exploratoria)

DES-CPL combina los siete modelos del benchmark. Los dos mejores benchmarks después de XGBoost, CatBoost y LightGBM, nunca entraron al conjunto. Como la búsqueda lleva el encogimiento a λ ≈ 1, DES-CPL funciona casi como un promedio ponderado, y un promedio puede mejorar si incorpora modelos fuertes y diversos. Aquí se añaden sus predicciones fuera de pliegue (3 pliegues, como el capítulo 13) y se repite la evaluación con 9 modelos.

Es una verificación **exploratoria**: el modelo con 9 modelos no es el pre-registrado. La regla fijada al inicio del capítulo exige una ventaja de al menos 0.006 frente a XGBoost para considerarla una vía real. Además, cualquier mejora aquí también la recibirían el apilamiento y los pesos globales con los mismos 9 modelos, y se reporta junto a ellos.

In [8]:
AMPLIADO = VERIF / "conjunto_ampliado"
AMPLIADO.mkdir(exist_ok=True)
SOTA = ("lightgbm", "catboost")
control_sota = []
for fold in FOLDS:
    ruta_sel, ruta_ev = AMPLIADO / f"seleccion_fold{fold}.csv", AMPLIADO / f"evaluacion_fold{fold}.csv"
    seleccion, evaluado = leer_fold(AMPLIADO if ruta_sel.exists() and ruta_ev.exists() else K3, fold)
    entrenamiento, evaluacion = particiones[fold]
    assert np.array_equal(seleccion["fila"], entrenamiento)
    X_tr, y_tr, g_tr = X.iloc[entrenamiento], y.iloc[entrenamiento], grupos.iloc[entrenamiento]
    X_ev = X.iloc[evaluacion]
    pliegues = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=cfg.SEMILLA)
                    .split(X_tr, y_tr, groups=g_tr))      # los mismos de ejecutar_fold
    for nombre in SOTA:
        if f"oof_{nombre}" not in seleccion:
            inicio = time.perf_counter()
            parametros = json.loads(sota.query("modelo == @nombre and fold == @fold")["parametros"].iloc[0])
            base = construir_pipeline_sota(nombre, roles).set_params(**parametros)
            oof, de_pliegue = np.full(len(entrenamiento), np.nan), np.zeros(len(evaluacion))
            for ajuste, validacion in pliegues:
                m = clone(base).fit(X_tr.iloc[ajuste], y_tr.iloc[ajuste])
                oof[validacion] = m.predict_proba(X_tr.iloc[validacion])[:, 1]
                de_pliegue += m.predict_proba(X_ev)[:, 1] / len(pliegues)
            seleccion[f"oof_{nombre}"] = oof
            evaluado[f"final_{nombre}"] = clone(base).fit(X_tr, y_tr).predict_proba(X_ev)[:, 1]
            evaluado[f"pliegue_{nombre}"] = de_pliegue
            exd._escribir(evaluado, ruta_ev)
            exd._escribir(seleccion, ruta_sel)
            print(f"[listo] fold {fold}, {nombre}: {time.perf_counter() - inicio:.0f} s")
        referencia = sota.query("modelo == @nombre and fold == @fold")["auc_pr"].iloc[0]
        control_sota.append({"fold": fold, "modelo": nombre, "AUC-PR benchmarks_sota": referencia,
                             "AUC-PR reajustado": average_precision_score(evaluado["y"], evaluado[f"final_{nombre}"])})
control_sota = pd.DataFrame(control_sota)
control_sota["diferencia"] = control_sota["AUC-PR reajustado"] - control_sota["AUC-PR benchmarks_sota"]
print("Control: máx. |diferencia| con results/benchmarks_sota.csv =",
      f"{control_sota['diferencia'].abs().max():.2e}")

Control: máx. |diferencia| con results/benchmarks_sota.csv = 1.17e-07


In [9]:
ampliado = evaluar_guardado(AMPLIADO, "evaluacion_ampliado", modelos=exd.MODELOS + SOTA)
METODOS_V4 = ["xgboost", "catboost", "lightgbm", "Pesos globales (Brier)", "Apilamiento logístico", "DES-CPL"]
tabla_v4 = pd.DataFrame({
    "7 modelos": [por_fold(k3, m).mean() if m in set(k3["método"]) else np.nan for m in METODOS_V4],
    "9 modelos": [por_fold(ampliado, m).mean() for m in METODOS_V4]}, index=METODOS_V4)
tabla_v4["diferencia"] = tabla_v4["9 modelos"] - tabla_v4["7 modelos"]
display(tabla_v4.round(4))
display(ampliado[ampliado["método"] == "DES-CPL"][["fold", "k", "tau", "lambda", "modelos_combinados",
                                                   "auc_pr_seleccion", "auc_pr"]].round(4))
pruebas_v4 = pd.DataFrame([comparar(ampliado, "DES-CPL", b=b) for b in ("xgboost", "catboost")]
                          + [comparar(ampliado, e, b="xgboost")
                             for e in ("Pesos globales (Brier)", "Apilamiento logístico")])
display(pruebas_v4.round(4))
fila_xgb = pruebas_v4.iloc[0]
v4_cumple = not (fila_xgb["dif. media"] >= 0.006 and fila_xgb["p (unilateral)"] < 0.05)
print("V4:", "el conjunto ampliado no abre una vía hacia la superioridad" if v4_cumple
      else "el conjunto ampliado SÍ supera a XGBoost con la ventaja exigida")

,7 modelos,9 modelos,diferencia
xgboost,0.2290,0.2290,0.0000
catboost,NaN,0.2285,NaN
lightgbm,NaN,0.2253,NaN
Pesos globales (Brier),0.2286,0.2296,0.0010
Apilamiento logístico,0.2280,0.2320,0.0040
DES-CPL,0.2279,0.2302,0.0023


,fold,k,tau,lambda,modelos_combinados,auc_pr_seleccion,auc_pr
12,0,20.0,0.0010,0.9,"random_forest, xgboost, lightgbm, catboost",0.2205,0.2530
25,1,100.0,0.0003,0.9,"random_forest, xgboost, svm, catboost",0.2268,0.2245
38,2,20.0,0.0030,0.9,"logistica, bayes, knn, arbol, random_forest, x...",0.2202,0.2142
51,3,20.0,0.0100,0.5,"random_forest, xgboost, lightgbm, catboost",0.2329,0.2244
64,4,50.0,0.0000,1.0,"logistica, bayes, knn, arbol, random_forest, x...",0.2328,0.2348


,comparación,dif. media,gana en folds,t (Nadeau-Bengio),p (unilateral)
0,DES-CPL − xgboost,0.0012,3/5,0.3905,0.3580
1,DES-CPL − catboost,0.0017,3/5,0.9594,0.1958
2,Pesos globales (Brier) − xgboost,0.0006,2/5,0.1734,0.4354
3,Apilamiento logístico − xgboost,0.0030,4/5,1.1200,0.1627


V4: el conjunto ampliado no abre una vía hacia la superioridad


## 20.6. V5: cinco pliegues en el conjunto de selección

Con 3 pliegues, cada modelo de pliegue se entrena con 2/3 del entrenamiento externo. Con 5 pliegues, con 4/5. DES-CPL mide la competencia con las predicciones de esos modelos (supuesto S2 del capítulo 15), y la curva de aprendizaje muestra que XGBoost es el más sensible al tamaño de muestra. Aquí se repite el cálculo completo del conjunto de selección con 5 pliegues, con los mismos folds externos, modelos e hiperparámetros, y se reevalúan todos los métodos.

Esta es la celda costosa (del orden de 8 a 12 minutos por fold externo). Guarda cada modelo en cuanto termina en `results/verificaciones/oof_k5/`.

In [10]:
K5 = VERIF / "oof_k5"
for fold in FOLDS:
    inicio = time.perf_counter()
    exd.ejecutar_fold(fold, particiones[fold], X, y, grupos, roles, mejores, K5,
                      razon_desbalance=razon, n_pliegues=5, verbose=False)
    print(f"Fold {fold}: {(time.perf_counter() - inicio) / 60:.1f} min")
print(exd.progreso(K5).to_string(index=False))
k5 = evaluar_guardado(K5, "evaluacion_k5")

Fold 0: 0.2 min


Fold 1: 0.2 min


Fold 2: 0.2 min


Fold 3: 0.2 min


Fold 4: 0.2 min
 fold  modelos listos faltan   estado
    0               7      — Completo
    1               7      — Completo
    2               7      — Completo
    3               7      — Completo
    4               7      — Completo


**Control.** El modelo final de cada fold no depende del número de pliegues, así que sus predicciones deben coincidir con las del capítulo 13. Lo único que cambia son las predicciones fuera de pliegue (`oof_`) y el promedio de los modelos de pliegue (`pliegue_`).

In [11]:
filas = []
for fold in FOLDS:
    (s3, e3), (s5, e5) = leer_fold(K3, fold), leer_fold(K5, fold)
    for m in exd.MODELOS:
        filas.append({"fold": fold, "modelo": m,
                      "máx. |final K5 − final K3|": np.abs(e5[f"final_{m}"] - e3[f"final_{m}"]).max(),
                      "AUC-PR fuera de pliegue K3": average_precision_score(s3["y"], s3[f"oof_{m}"]),
                      "AUC-PR fuera de pliegue K5": average_precision_score(s5["y"], s5[f"oof_{m}"]),
                      "|logit pliegue − logit final| K3": np.abs(logit(e3[f"pliegue_{m}"]) - logit(e3[f"final_{m}"])).mean(),
                      "|logit pliegue − logit final| K5": np.abs(logit(e5[f"pliegue_{m}"]) - logit(e5[f"final_{m}"])).mean()})
pliegues_modelos = pd.DataFrame(filas)
print("Control: máx. |final K5 − final K3| =", f"{pliegues_modelos['máx. |final K5 − final K3|'].max():.2e}")
resumen_modelos = pliegues_modelos.groupby("modelo").mean(numeric_only=True).drop(columns=["fold", "máx. |final K5 − final K3|"])
resumen_modelos["ganancia fuera de pliegue"] = (resumen_modelos["AUC-PR fuera de pliegue K5"]
                                                - resumen_modelos["AUC-PR fuera de pliegue K3"])
display(resumen_modelos.sort_values("ganancia fuera de pliegue", ascending=False).round(4))

Control: máx. |final K5 − final K3| = 0.00e+00


,AUC-PR fuera de pliegue K3,AUC-PR fuera de pliegue K5,|logit pliegue − logit final| K3,|logit pliegue − logit final| K5,ganancia fuera de pliegue
modelo,,,,,
xgboost,0.2179,0.2228,0.1808,0.1299,0.0049
knn,0.2019,0.2031,0.1036,0.0782,0.0012
random_forest,0.2203,0.2213,0.0295,0.0258,0.0010
arbol,0.2055,0.2061,0.1545,0.1277,0.0006
svm,0.2135,0.2139,0.0094,0.0053,0.0004
logistica,0.2103,0.2107,0.0183,0.0102,0.0003
bayes,0.1991,0.1994,0.0755,0.0406,0.0003


La tabla anterior mide el sesgo del supuesto S2 en cada modelo. La columna «ganancia fuera de pliegue» indica cuánto mejora la predicción fuera de pliegue al pasar de 2/3 a 4/5 de los datos. Las columnas de |logit pliegue − logit final| indican cuánto se alejan los modelos de pliegue del modelo final. La siguiente compara los métodos.

In [12]:
METODOS_V5 = list(exd.MODELOS) + ["Promedio simple", "Apilamiento logístico", "Pesos globales (Brier)", "DES-CPL"]
tabla_v5 = pd.DataFrame({"AUC-PR K3": [por_fold(k3, m).mean() for m in METODOS_V5],
                         "AUC-PR K5": [por_fold(k5, m).mean() for m in METODOS_V5],
                         "AUC-PR selección K3": [por_fold(k3, m, "auc_pr_seleccion").mean() for m in METODOS_V5],
                         "AUC-PR selección K5": [por_fold(k5, m, "auc_pr_seleccion").mean() for m in METODOS_V5]},
                        index=METODOS_V5)
tabla_v5["Δ evaluación (K5 − K3)"] = tabla_v5["AUC-PR K5"] - tabla_v5["AUC-PR K3"]
display(tabla_v5.round(4))

configuracion = pd.concat({f"K{k}": t[t["método"] == "DES-CPL"].set_index("fold")[["k", "tau", "lambda", "modelos_combinados", "auc_pr"]]
                           for k, t in ((3, k3), (5, k5))}, axis=1)
display(configuracion)

pruebas_v5 = pd.DataFrame([{**comparar(t, "DES-CPL", b=b), "pliegues": k}
                           for k, t in ((3, k3), (5, k5))
                           for b in ("xgboost", "Pesos globales (Brier)")])
display(pruebas_v5.round(4))
delta_v5 = tabla_v5.loc["DES-CPL", "Δ evaluación (K5 − K3)"]
frente_xgb = pruebas_v5[pruebas_v5["comparación"] == "DES-CPL − xgboost"].set_index("pliegues")["p (unilateral)"]
p3, p5 = frente_xgb[3], frente_xgb[5]
v5_cumple = bool(abs(delta_v5) < 0.002 and ((p3 < 0.05) == (p5 < 0.05)))
print(f"DES-CPL: Δ AUC-PR (K5 − K3) = {delta_v5:+.4f}; Nadeau-Bengio frente a XGBoost: p = {p3:.3f} (K3), {p5:.3f} (K5)")
print("V5:", "el número de pliegues no cambia la conclusión" if v5_cumple
      else "el número de pliegues SÍ cambia la conclusión")

,AUC-PR K3,AUC-PR K5,AUC-PR selección K3,AUC-PR selección K5,Δ evaluación (K5 − K3)
logistica,0.2109,0.2109,0.2103,0.2107,0.0000
bayes,0.1997,0.1997,0.1991,0.1994,0.0000
knn,0.2065,0.2065,0.2019,0.2031,0.0000
arbol,0.2021,0.2021,0.2055,0.2061,0.0000
random_forest,0.2237,0.2237,0.2203,0.2213,0.0000
xgboost,0.2290,0.2290,0.2179,0.2228,0.0000
svm,0.2145,0.2145,0.2135,0.2139,0.0000
Promedio simple,0.2185,0.2185,0.2162,0.2172,0.0000
Apilamiento logístico,0.2280,0.2291,0.2226,0.2258,0.0011
Pesos globales (Brier),0.2286,0.2304,0.2227,0.2259,0.0018


K3                                                                                 K5                                                                            
          k     tau lambda                                 modelos_combinados    auc_pr      k     tau lambda                                 modelos_combinados    auc_pr
fold                                                                                                                                                                      
0      30.0  0.0001    1.0             logistica, random_forest, xgboost, svm  0.256232   50.0  0.0001   1.00             logistica, random_forest, xgboost, svm  0.256245
1      50.0  0.0003    1.0             logistica, random_forest, xgboost, svm  0.218907  100.0  0.0010   0.75             logistica, random_forest, xgboost, svm  0.218959
2      20.0  0.0001    1.0             logistica, random_forest, xgboost, svm  0.212791   50.0  0.0003   0.90             logistica, random_forest, xgboost, svm  0.221475
3      20.0  0.0100    0.5  logistica, bayes, knn, arbol, random_forest, x...  0.219186   50.0  0.0010   0.90  logistica, bayes, knn, arbol, random_forest, x...  0.219169
4     100.0  0.0003    0.9  logistica, bayes, knn, arbol, random_forest, x...  0.232506   10.0  0.0010   0.90  logistica, bayes, knn, arbol, random_forest, x...  0.228614

,comparación,dif. media,gana en folds,t (Nadeau-Bengio),p (unilateral),pliegues
0,DES-CPL − xgboost,-0.0010,1/5,-0.2622,0.5969,3
1,DES-CPL − Pesos globales (Brier),-0.0007,1/5,-0.7817,0.7610,3
2,DES-CPL − xgboost,-0.0001,2/5,-0.0167,0.5063,5
3,DES-CPL − Pesos globales (Brier),-0.0015,2/5,-0.9742,0.8074,5


DES-CPL: Δ AUC-PR (K5 − K3) = +0.0010; Nadeau-Bengio frente a XGBoost: p = 0.597 (K3), 0.506 (K5)
V5: el número de pliegues no cambia la conclusión


## 20.7. V6: predicción conforme con 3 y 5 pliegues

En la sección 13.8 el conjunto conforme de DES-CPL cubrió al 89.0 % de todos los encuentros, pero solo al 86.9 % de los readmitidos, por debajo del 90 % nominal. La explicación propuesta fue que la calibración usa la predicción fuera de pliegue, mientras que la prueba usa la del modelo reentrenado, y las dos no son intercambiables. Si es así, con 5 pliegues, cuyos modelos se parecen más al final, la cobertura debería acercarse al 90 %. Aquí se repite el procedimiento en cada fold externo: se calibra con la predicción fuera de pliegue del conjunto de selección y se mide en el fold de evaluación (α = 0.1, Mondrian por clase).

In [13]:
filas = []
for k in (3, 5):
    for fold in FOLDS:
        cal = pd.read_csv(VERIF / f"evaluacion_k{k}_calibracion_fold{fold}.csv")
        ev = pd.read_csv(VERIF / f"evaluacion_k{k}_predicciones_fold{fold}.csv")
        cobertura, resumen = evaluar_conformal(cal["y"], cal["DES-CPL"], ev["y"], ev["DES-CPL"], alfa=0.1)
        cob = cobertura.set_index("grupo")["cobertura"]
        filas.append({"pliegues": k, "fold": fold, "todos": cob["Todos"],
                      "no readmitidos": cob["No readmitidos"], "readmitidos": cob["Readmitidos"],
                      "conjuntos con ambas clases": resumen["conjuntos_con_ambas"]})
conforme = pd.DataFrame(filas)
conforme.to_csv(VERIF / "conformal_pliegues.csv", index=False)
display(conforme.groupby("pliegues").mean(numeric_only=True).drop(columns="fold").round(4))

,todos,no readmitidos,readmitidos,conjuntos con ambas clases
pliegues,,,,
3,0.8976,0.8978,0.8957,0.6397
5,0.8976,0.8982,0.8928,0.6374


## 20.8. V7: parada temprana de XGBoost

`XGBClasificadorParada` separa el 10 % de lo que recibe para decidir cuándo parar el boosting. Esa separación tiene dos problemas, ya declarados en su docstring:
1. no está agrupada por paciente, así que encuentros de un mismo paciente quedan a ambos lados;
2. con ADASYN, que es el balanceo de XGBoost, se hace después del remuestreo, así que la validación contiene muestras sintéticas interpoladas del resto.

Las dos cosas hacen que la curva de validación parezca mejor durante más tiempo y que el modelo pare más tarde. La cota estimada en el benchmark (docstring de `XGBClasificadorParada`) era de 0.0024 de AUC-PR. Aquí se mide directamente con la configuración de 60 evaluaciones. Se reajusta el XGBoost final de cada fold de dos formas:
- **original:** reproduce el capítulo 13 (control);
- **corregida:** la validación es un 10 % de *pacientes* reales (`StratifiedGroupKFold` de 10 pliegues, se toma uno) y ADASYN se aplica solo al resto.

El preprocesador se ajusta en ambas con todo el fold de entrenamiento, como en el original, para que solo cambie la parada.

**Regla (fijada antes de ejecutar).** La conclusión se mantiene si la AUC-PR media de XGBoost cambia menos que la cota declarada (0.0024) y la prueba de Nadeau-Bengio de DES-CPL frente al XGBoost corregido no cambia de veredicto a α = 0.05. DES-CPL se mantiene fijo, con las predicciones del capítulo 13: no se recalcula con el XGBoost corregido.

In [14]:
ruta = VERIF / "xgboost_parada.csv"
if ruta.exists():
    xgb_parada = pd.read_csv(ruta)
else:
    fila, filas = mejores.loc["xgboost"], []
    for fold in FOLDS:
        inicio = time.perf_counter()
        entrenamiento, evaluacion = particiones[fold]
        _, evaluado = leer_fold(K3, fold)
        assert np.array_equal(evaluado["fila"], evaluacion)
        X_tr, X_ev = X.iloc[entrenamiento], X.iloc[evaluacion]
        y_tr, g_tr = y.iloc[entrenamiento].to_numpy(), grupos.iloc[entrenamiento].to_numpy()
        base = construir_pipeline("xgboost", fila["balanceo"], roles, cfg.SEMILLA, razon).set_params(
            **fila["hiperparametros_por_fold"][fold])

        # Original: el pipeline tal cual (reproduce el capítulo 13).
        original = clone(base).fit(X_tr, y_tr)
        p_original = original.predict_proba(X_ev)[:, 1]

        # Corregida: validación de pacientes reales, separada antes del remuestreo.
        ajuste, validacion = next(StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=cfg.SEMILLA)
                                  .split(X_tr, y_tr, groups=g_tr))
        pre = clone(base.named_steps["preprocesamiento"]).fit(X_tr, y_tr)
        X_aj, y_aj = pre.transform(X_tr.iloc[ajuste]), y_tr[ajuste]
        if "balanceo" in base.named_steps:
            X_aj, y_aj = clone(base.named_steps["balanceo"]).fit_resample(X_aj, y_aj)
        corregido = base.named_steps["modelo"]._crear()
        corregido.fit(X_aj, y_aj, eval_set=[(pre.transform(X_tr.iloc[validacion]), y_tr[validacion])],
                      verbose=False)
        p_corregida = corregido.predict_proba(pre.transform(X_ev))[:, 1]

        filas.append({"fold": fold,
                      "control: máx. |p original − p guardada|": np.abs(p_original - evaluado["final_xgboost"]).max(),
                      "AUC-PR original": average_precision_score(evaluado["y"], p_original),
                      "AUC-PR corregida": average_precision_score(evaluado["y"], p_corregida),
                      "árboles original": original.named_steps["modelo"].mejor_iteracion_ + 1,
                      "árboles corregida": int(corregido.best_iteration) + 1,
                      "pacientes de validación": len(np.unique(g_tr[validacion]))})
        print(f"[listo] fold {fold}: {time.perf_counter() - inicio:.0f} s")
    xgb_parada = pd.DataFrame(filas)
    xgb_parada["diferencia"] = xgb_parada["AUC-PR corregida"] - xgb_parada["AUC-PR original"]
    xgb_parada.to_csv(ruta, index=False)
display(xgb_parada.round(4))

des = por_fold(k3, "DES-CPL").to_numpy()
pruebas_v7 = pd.DataFrame([
    {"XGBoost": nombre, "AUC-PR XGBoost": a.mean(), "AUC-PR DES-CPL": des.mean(),
     **{c: v for c, v in comparar_arrays(des, a).items()}}
    for nombre, a in (("original", xgb_parada["AUC-PR original"].to_numpy()),
                      ("corregida", xgb_parada["AUC-PR corregida"].to_numpy()))])
display(pruebas_v7.round(4))
p_orig, p_corr = pruebas_v7["p (unilateral)"]
v7_cumple = bool(abs(xgb_parada["diferencia"].mean()) < 0.0024 and ((p_orig < 0.05) == (p_corr < 0.05)))
print(f"Control: máx. |p original − p guardada| = {xgb_parada['control: máx. |p original − p guardada|'].max():.1e}")
print(f"XGBoost corregido − original: {xgb_parada['diferencia'].mean():+.4f} de media; árboles "
      f"{xgb_parada['árboles original'].mean():.0f} → {xgb_parada['árboles corregida'].mean():.0f}")
print("V7:", "la parada temprana no cambia la conclusión" if v7_cumple
      else "la parada temprana SÍ cambia la conclusión")

[listo] fold 0: 88 s


[listo] fold 1: 89 s


[listo] fold 2: 111 s


[listo] fold 3: 88 s


[listo] fold 4: 98 s


,fold,control: máx. |p original − p guardada|,AUC-PR original,AUC-PR corregida,árboles original,árboles corregida,pacientes de validación,diferencia
0,0,0.0,0.2478,0.2163,219,47,4470,-0.0315
1,1,0.0,0.2193,0.2142,139,179,4516,-0.0050
2,2,0.0,0.2203,0.1139,898,1,4456,-0.1064
3,3,0.0,0.2222,0.2154,156,81,4497,-0.0068
4,4,0.0,0.2352,0.1139,494,1,4495,-0.1214


,XGBoost,AUC-PR XGBoost,AUC-PR DES-CPL,dif. media,gana en folds,t (Nadeau-Bengio),p (unilateral)
0,original,0.2290,0.2279,-0.0010,1/5,-0.2622,0.5969
1,corregida,0.1747,0.2279,0.0532,5/5,1.4887,0.1054


Control: máx. |p original − p guardada| = 5.0e-09
XGBoost corregido − original: -0.0542 de media; árboles 381 → 62
V7: la parada temprana SÍ cambia la conclusión


## 20.9. V8: calibración del SVM agrupada por paciente

El SVM es un `LinearSVC` envuelto en `CalibratedClassifierCV(cv=3)`, que ajusta el SVM en dos tercios y la sigmoide de calibración en el tercio restante. Esos tercios son estratificados, pero no agrupados por paciente. V2 mostró que su orden no importa. Aquí se comprueba si importa que un mismo paciente quede a ambos lados: se reajusta el SVM final de cada fold con pliegues de calibración agrupados por paciente y se comparan su AUC-PR y su Brier.

**Regla (fijada antes de ejecutar).** La conclusión se mantiene si la AUC-PR media cambia menos de 0.002.

In [15]:
ruta = VERIF / "svm_calibracion_agrupada.csv"
if ruta.exists():
    svm_agrupado = pd.read_csv(ruta)
else:
    fila, filas = mejores.loc["svm"], []
    # Los pliegues de calibración se pasan como índices de filas: exige que el
    # pipeline no remuestree (no lo hace con estos balanceos).
    assert fila["balanceo"] in ("ninguno", "class_weight")
    for fold in FOLDS:
        entrenamiento, evaluacion = particiones[fold]
        _, evaluado = leer_fold(K3, fold)
        X_tr, y_tr, g_tr = X.iloc[entrenamiento], y.iloc[entrenamiento], grupos.iloc[entrenamiento]
        base = construir_pipeline("svm", fila["balanceo"], roles, cfg.SEMILLA, razon).set_params(
            **fila["hiperparametros_por_fold"][fold])
        pliegues_cal = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=cfg.SEMILLA)
                            .split(X_tr, y_tr, groups=g_tr))
        agrupado = clone(base).set_params(modelo__cv=pliegues_cal)
        p = agrupado.fit(X_tr, y_tr).predict_proba(X.iloc[evaluacion])[:, 1]
        filas.append({"fold": fold,
                      "AUC-PR sin agrupar": average_precision_score(evaluado["y"], evaluado["final_svm"]),
                      "AUC-PR agrupada": average_precision_score(evaluado["y"], p),
                      "Brier sin agrupar": np.mean((evaluado["final_svm"] - evaluado["y"]) ** 2),
                      "Brier agrupada": np.mean((p - evaluado["y"]) ** 2)})
    svm_agrupado = pd.DataFrame(filas)
    svm_agrupado["diferencia AUC-PR"] = svm_agrupado["AUC-PR agrupada"] - svm_agrupado["AUC-PR sin agrupar"]
    svm_agrupado.to_csv(ruta, index=False)
v8_cumple = bool(abs(svm_agrupado["diferencia AUC-PR"].mean()) < 0.002)
display(svm_agrupado.round(4))
print(f"Diferencia media de AUC-PR {svm_agrupado['diferencia AUC-PR'].mean():+.4f}; "
      f"de Brier {(svm_agrupado['Brier agrupada'] - svm_agrupado['Brier sin agrupar']).mean():+.5f}")
print("V8:", "agrupar la calibración no cambia al SVM" if v8_cumple else "agrupar la calibración SÍ cambia al SVM")

,fold,AUC-PR sin agrupar,AUC-PR agrupada,Brier sin agrupar,Brier agrupada,diferencia AUC-PR
0,0,0.2373,0.2374,0.0958,0.0958,0.0001
1,1,0.2054,0.2055,0.0977,0.0977,0.0001
2,2,0.2091,0.2092,0.0974,0.0974,0.0001
3,3,0.2058,0.2063,0.0976,0.0977,0.0005
4,4,0.2147,0.2160,0.0971,0.0970,0.0013


Diferencia media de AUC-PR +0.0004; de Brier -0.00001
V8: agrupar la calibración no cambia al SVM


## 20.10. V9: registros duplicados en los conjuntos adicionales

Los conjuntos adicionales no tienen un identificador de persona, así que cada registro es su propio grupo. Si un conjunto tiene registros con las mismas variables, una copia puede quedar en el entrenamiento y otra en la evaluación. Primero se cuenta, en cada conjunto y fold, qué fracción de la evaluación tiene una copia exacta de sus variables en el entrenamiento.

Hay que interpretarlo con cuidado. En CDC las 21 variables son binarias u ordinales, así que dos personas distintas pueden responder exactamente igual. Una copia no es necesariamente el mismo registro repetido, y esos perfiles comunes también aparecen al aplicar el modelo. Quitarlos de la evaluación tampoco es neutral: deja solo los perfiles poco frecuentes. Por eso se reporta como sensibilidad, no como corrección. En los conjuntos donde más del 1 % de la evaluación tiene copia, se recalculan las métricas de cada método sin esos registros. Las predicciones de los modelos base y de DES-CPL salen de las guardadas en `results/adicionales/`; LightGBM y CatBoost se reajustan con su configuración guardada.

**Regla (fijada antes de ejecutar).** La conclusión del capítulo 16 se mantiene si, sin los registros con copia, las diferencias de DES-CPL frente a XGBoost y frente al mejor benchmark de cada conjunto conservan su signo y cambian menos de 0.002.

In [16]:
import importlib.util

espec = importlib.util.spec_from_file_location("conjuntos_adicionales",
                                               RAIZ / "experiments" / "conjuntos_adicionales.py")
ca = importlib.util.module_from_spec(espec)
espec.loader.exec_module(ca)

adicionales, filas, resumen_dup = {}, [], []
for nombre in ca.ORDEN:
    Xa, ya, roles_a = ca.preparar(nombre)
    particiones_a = exd.particiones_externas(Xa, ya, pd.Series(np.arange(len(Xa))), 5, cfg.SEMILLA)
    clave = pd.util.hash_pandas_object(Xa, index=False).to_numpy()
    adicionales[nombre] = (Xa, ya, roles_a, particiones_a, clave)
    perfiles = pd.DataFrame({"clave": clave, "y": ya}).groupby("clave")["y"].agg(["size", "nunique"])
    repetidos = perfiles[perfiles["size"] > 1]
    resumen_dup.append({"conjunto": nombre, "registros": len(Xa),
                        "registros con otro idéntico": float(pd.Series(clave).duplicated(keep=False).mean()),
                        "perfiles repetidos con etiquetas distintas": float((repetidos["nunique"] > 1).mean())
                        if len(repetidos) else np.nan})
    for fold, (tr, ev) in enumerate(particiones_a):
        filas.append({"conjunto": nombre, "fold": fold,
                      "evaluación con copia en entrenamiento": float(np.isin(clave[ev], clave[tr]).mean())})
copias = pd.DataFrame(filas)
display(pd.DataFrame(resumen_dup).round(4))
display(copias.groupby("conjunto")["evaluación con copia en entrenamiento"].agg(["mean", "max"]).round(4))
con_copias = [n for n in ca.ORDEN
              if copias.query("conjunto == @n")["evaluación con copia en entrenamiento"].mean() > 0.01]
print("Conjuntos con más del 1 % de la evaluación copiada:", con_copias or "ninguno")

,conjunto,registros,registros con otro idéntico,perfiles repetidos con etiquetas distintas
0,credito,30000,0.0036,0.4038
1,banco,45211,0.0007,0.0000
2,cdc,100000,0.0977,0.0997


,mean,max
conjunto,,
banco,0.0006,0.0009
cdc,0.0870,0.0885
credito,0.0028,0.0033


Conjuntos con más del 1 % de la evaluación copiada: ['cdc']


In [17]:
ruta = VERIF / "duplicados_sensibilidad.csv"
sens = pd.read_csv(ruta) if ruta.exists() else pd.DataFrame(columns=["conjunto", "fold"])
for nombre in con_copias:
    Xa, ya, roles_a, particiones_a, clave = adicionales[nombre]
    carpeta = RESULTADOS / "adicionales" / nombre
    configs = json.loads((carpeta / "configuraciones.json").read_text())
    for fold, (tr, ev) in enumerate(particiones_a):
        if ((sens["conjunto"] == nombre) & (sens["fold"] == fold)).any():
            continue
        inicio = time.perf_counter()
        seleccion, evaluado = leer_fold(carpeta, fold)
        assert np.array_equal(evaluado["fila"], ev)
        _, pred = exd.evaluar_fold(seleccion, evaluado, list(exd.MODELOS), semilla=cfg.SEMILLA + fold,
                                   variantes=("DES-CPL",), devolver_predicciones=True)
        pred.pop("DES-CPL (fuera de pliegue)", None)
        for m in SOTA:
            est = clone(construir_pipeline_sota(m, roles_a)).set_params(**configs[f"{fold}|{m}"])
            pred[m] = est.fit(Xa.iloc[tr], ya.iloc[tr]).predict_proba(Xa.iloc[ev])[:, 1]
        y_ev = evaluado["y"].to_numpy()
        sin_copia = ~np.isin(clave[ev], clave[tr])
        filas = [{"conjunto": nombre, "fold": fold, "método": m,
                  "AUC-PR todos": average_precision_score(y_ev, p),
                  "AUC-PR sin copia": average_precision_score(y_ev[sin_copia], np.asarray(p)[sin_copia]),
                  "n sin copia": int(sin_copia.sum())} for m, p in pred.items()]
        sens = pd.concat([sens, pd.DataFrame(filas)], ignore_index=True)
        sens.to_csv(ruta, index=False)
        print(f"[listo] {nombre}, fold {fold}: {time.perf_counter() - inicio:.0f} s")

v9_cumple = True
if len(sens):
    # Control: con todos los registros se reproducen las cifras del capítulo 16.
    oficial_ad = pd.read_csv(RESULTADOS / "adicionales" / "evaluacion_conjuntos.csv")
    cruce = sens.merge(oficial_ad[["conjunto", "fold", "método", "auc_pr"]], on=["conjunto", "fold", "método"])
    print(f"Control: máx. |AUC-PR todos − capítulo 16| = {(cruce['AUC-PR todos'] - cruce['auc_pr']).abs().max():.1e}"
          f" en {len(cruce)} filas")
    medias = sens.groupby(["conjunto", "método"])[["AUC-PR todos", "AUC-PR sin copia"]].mean()
    filas = []
    for nombre in sens["conjunto"].unique():
        m = medias.loc[nombre]
        benchmarks = [b for b in list(exd.MODELOS) + list(SOTA) if b in m.index]
        mejor = m.loc[benchmarks, "AUC-PR todos"].idxmax()
        for rival in dict.fromkeys(["xgboost", mejor]):
            antes = m.loc["DES-CPL", "AUC-PR todos"] - m.loc[rival, "AUC-PR todos"]
            despues = m.loc["DES-CPL", "AUC-PR sin copia"] - m.loc[rival, "AUC-PR sin copia"]
            filas.append({"conjunto": nombre, "DES-CPL frente a": rival, "dif. todos": antes,
                          "dif. sin copia": despues, "cambio": despues - antes})
    tabla_v9 = pd.DataFrame(filas)
    v9_cumple = bool(((np.sign(tabla_v9["dif. todos"]) == np.sign(tabla_v9["dif. sin copia"]))
                      & (tabla_v9["cambio"].abs() < 0.002)).all())
    display(medias.round(4))
    display(tabla_v9.round(4))
print("V9:", "los duplicados no cambian la conclusión" if v9_cumple else "los duplicados SÍ cambian la conclusión")

[listo] cdc, fold 0: 144 s


[listo] cdc, fold 1: 145 s


[listo] cdc, fold 2: 138 s


[listo] cdc, fold 3: 102 s


[listo] cdc, fold 4: 100 s
Control: máx. |AUC-PR todos − capítulo 16| = 2.5e-04 en 65 filas


AUC-PR todos  AUC-PR sin copia
conjunto método                                                
cdc      Apilamiento logístico         0.4272            0.4313
         DES-CPL                       0.4258            0.4299
         Pesos globales (Brier)        0.4256            0.4297
         Promedio simple               0.4157            0.4198
         arbol                         0.3928            0.3969
         bayes                         0.3489            0.3519
         catboost                      0.4236            0.4278
         knn                           0.3883            0.4133
         lightgbm                      0.4250            0.4291
         logistica                     0.4019            0.4055
         random_forest                 0.4208            0.4250
         svm                           0.4018            0.4054
         xgboost                       0.4199            0.4241

,conjunto,DES-CPL frente a,dif. todos,dif. sin copia,cambio
0,cdc,xgboost,0.0060,0.0058,-0.0002
1,cdc,lightgbm,0.0009,0.0008,-0.0001


V9: los duplicados no cambian la conclusión


## 20.11. Síntesis

In [18]:
sintesis = pd.DataFrame([
    {"verificación": "V1 Reproducibilidad", "se mantiene la conclusión": v1_cumple},
    {"verificación": "V2 Orden de las filas en el SVM", "se mantiene la conclusión": v2_cumple},
    {"verificación": "V3 Reproducción de Bayes", "se mantiene la conclusión": v3_cumple},
    {"verificación": "V4 Conjunto ampliado (exploratoria)", "se mantiene la conclusión": v4_cumple},
    {"verificación": "V5 Cinco pliegues en la selección", "se mantiene la conclusión": v5_cumple},
    {"verificación": "V7 Parada temprana de XGBoost", "se mantiene la conclusión": v7_cumple},
    {"verificación": "V8 Calibración del SVM agrupada", "se mantiene la conclusión": v8_cumple},
    {"verificación": "V9 Duplicados en los conjuntos adicionales", "se mantiene la conclusión": v9_cumple},
])
sintesis["se mantiene la conclusión"] = sintesis["se mantiene la conclusión"].map({True: "Sí", False: "No"})
sintesis.to_csv(VERIF / "sintesis.csv", index=False)
display(sintesis)

,verificación,se mantiene la conclusión
0,V1 Reproducibilidad,Sí
1,V2 Orden de las filas en el SVM,Sí
2,V3 Reproducción de Bayes,No
3,V4 Conjunto ampliado (exploratoria),Sí
4,V5 Cinco pliegues en la selección,Sí
5,V7 Parada temprana de XGBoost,No
6,V8 Calibración del SVM agrupada,Sí
7,V9 Duplicados en los conjuntos adicionales,Sí


### 20.11.1. Lectura

**V1. Reproducibilidad: se cumple.** Al evaluar de nuevo las predicciones guardadas se obtienen las mismas métricas y configuraciones del capítulo 13 en los 11 métodos y los 5 folds (diferencia máxima 1e-16). También confirma que omitir las ablaciones no cambia el resultado de DES-CPL.

**V2. Orden de las filas en el SVM: se cumple.** El problema principal no tiene la deriva de Bank Marketing. La prevalencia apenas cambia a lo largo de las filas (11.7 %, 11.6 % y 10.9 % por tercio). Con las filas barajadas, el SVM cambia en promedio +0.0004 de AUC-PR, con un máximo de 0.0014, mientras que en Bank Marketing la caída fue de 0.30. Los resultados del SVM, y los de DES-CPL que lo incluye, son válidos.

**V3. Reproducción de Bayes: la regla escrita no se cumple, pero la diferencia queda explicada y no tiene efecto.**
- *Causa.* Al reajustarlo hoy, Bayes reproduce exactamente la AUC-PR del capítulo 12 en los 5 folds, tanto con `src/` como con `experimento.py`. Sus probabilidades coinciden también con las guardadas en el capítulo 13 (diferencia < 1e-8). La diferencia aparece al guardar. `_escribir` escribe las predicciones con 8 cifras significativas (`float_format="%.8g"`), y Bayes es el único modelo con probabilidades tan cercanas a 1: entre 65 y 99 por fold quedan como 1.0 exacto. Esos empates cambian la AUC-PR hasta en 8e-4. Es un artefacto de almacenamiento, no del modelo.
- *Efecto en DES-CPL.* Ninguno. Las probabilidades cambian menos de 5e-9 y el perfil recorta en 1e-6, así que la región de competencia es la misma.
- *Por qué falló la regla.* Quitar a Bayes no resultó ser una cota de su influencia. DES-CPL cambió hasta 0.0017 en los folds 0 a 2, donde Bayes ni siquiera está entre los modelos combinados, porque cambiar el conjunto de modelos cambia el espacio que recorre la búsqueda de Optuna. Lo que midió es la variabilidad de la propia búsqueda, de entre 0.001 y 0.004 por fold, no el efecto de Bayes.

**V4. Conjunto ampliado (exploratoria): se cumple, no abre una vía.**
- Con LightGBM y CatBoost, DES-CPL sube de 0.2279 a 0.2302 y queda 0.0012 sobre XGBoost (gana en 3 de 5 folds, p = 0.36). Le falta mucho para la ventaja de 0.006 que haría falta.
- El que más gana es el apilamiento: 0.2320, +0.0030 sobre XGBoost, en 4 de 5 folds (p = 0.16). La mejora viene de combinar más modelos fuertes, no de seleccionarlos localmente, y un ensamble estático la aprovecha mejor.

**V5. Cinco pliegues en el conjunto de selección: se cumple.**
- El modelo final es idéntico con 3 y con 5 pliegues (diferencia 0), así que solo cambian las predicciones fuera de pliegue.
- Se confirma la asimetría del supuesto S2. XGBoost es el modelo que más se beneficia de entrenar con 4/5 de los datos: su AUC-PR fuera de pliegue sube 0.0049, frente a 0.0012 o menos en los demás. Además, la distancia media entre su modelo de pliegue y el final baja de 0.18 a 0.13 en logit.
- Aun así, DES-CPL solo cambia +0.0010 (0.2279 a 0.2289). Las λ elegidas siguen entre 0.75 y 1, y frente a XGBoost la diferencia pasa de −0.0010 a −0.0001 (p = 0.60 y 0.51). Los pesos globales llegan a 0.2304, 0.0014 sobre XGBoost en media (no se probó su significancia en este capítulo).
- Usar 3 pliegues no decidió la conclusión.

**V6. Predicción conforme: la explicación de la sección 13.8 no se sostiene.** En los folds externos, la calibración también usa predicciones fuera de pliegue y la evaluación usa el modelo final. Aun así, la cobertura de los readmitidos es 0.896 con 3 pliegues y 0.893 con 5, prácticamente el 90 % nominal, y pasar a 5 pliegues no la mejora. La diferencia entre los modelos de pliegue y el final no produce, entonces, la subcobertura. El 0.869 del conjunto de prueba sigue sin explicar: con unos 2 263 readmitidos son unos 5 errores estándar, así que no parece azar. Lo esperable es que dependa de esa partición concreta.

**V7. Parada temprana de XGBoost: la regla no se cumple, pero en la dirección que no compromete la conclusión.**
- El control reproduce exactamente el XGBoost del capítulo 13.
- La versión corregida, con la validación tomada de pacientes reales antes de ADASYN, es peor en los 5 folds: −0.054 de AUC-PR en media.
- En los folds 2 y 4 la parada se dispara en el primer árbol: las predicciones quedan prácticamente constantes (AUC-PR 0.1139, igual a la prevalencia). En los otros tres folds también pierde (−0.032, −0.005 y −0.007).
- La parada original termina mucho más tarde (381 árboles en media, frente a 62). En este problema, parar tarde es mejor. La causa del empeoramiento no está verificada: que el modelo se quede con un solo árbol indica que el criterio de parada interno falla con la nueva validación. Una corrección completa exigiría repetir la búsqueda de hiperparámetros con la nueva parada.
- La fuga de la parada original ocurre dentro del entrenamiento. Por eso la AUC-PR de 0.2290 en el bucle externo sigue siendo una estimación válida de ese pipeline, y el XGBoost con que se comparó DES-CPL es su versión más fuerte, no una debilitada.
- No sería correcto usar el XGBoost corregido para afirmar que DES-CPL gana (+0.053, p = 0.11): sería compararlo con un modelo roto. Una corrección completa exigiría repetir la búsqueda de XGBoost con la nueva parada (unos 70 minutos). No cambiaría la conclusión salvo que XGBoost mejorara, lo que solo reforzaría que DES-CPL no lo supera.

**V8. Calibración del SVM agrupada: se cumple.** Con los pliegues de calibración agrupados por paciente, la AUC-PR cambia +0.0004 en media y el Brier no cambia (−0.00001).

**V9. Duplicados en los conjuntos adicionales: se cumple.**
- En Crédito (0.3 %) y Bank (0.1 %) casi ningún registro de evaluación tiene una copia en el entrenamiento.
- En CDC la tiene el 8.7 %. El 10 % de los perfiles repetidos tiene etiquetas distintas, lo que indica personas distintas con las mismas respuestas y no registros duplicados.
- Sin esos registros, todos los métodos suben unos 0.004 (k-NN, 0.025). Las diferencias de DES-CPL casi no cambian: frente a XGBoost pasan de +0.0060 a +0.0058 y frente a LightGBM, el mejor benchmark de CDC, de +0.0009 a +0.0008.
- El control difiere del capítulo 16 en hasta 2.5e-4, solo en Bayes y en los métodos que lo combinan. Es el mismo redondeo de V3: el capítulo 16 evaluó las predicciones en memoria y aquí se leen del CSV.

**En conjunto.** Ninguna de las nueve verificaciones cambia el veredicto: DES-CPL sigue sin superar a XGBoost con significancia (R3 no se cumple). Las dos reglas que no se cumplen (V3 y V7) quedan explicadas. Ninguna de las dos favorece a DES-CPL: V3 es un artefacto de almacenamiento sin efecto, y V7 muestra que el XGBoost de referencia es la versión más fuerte disponible. Las consecuencias para el resto del trabajo, ya aplicadas en los capítulos 13 y 17 y en el artículo, son:
1. explicar la diferencia de Bayes del capítulo 13 como un efecto del redondeo al guardar;
2. retirar la explicación por intercambiabilidad de la subcobertura conforme en las secciones 13.8 y 17.3, y en la ficha del modelo;
3. citar este capítulo como respuesta a la objeción de los 3 pliegues;
4. declarar en las limitaciones que la parada temprana de XGBoost usa una validación no agrupada y con muestras sintéticas, y que corregirla sin repetir la búsqueda empeora a XGBoost.